In [ ]:
!pip install transformers datasets evaluate rouge_score accelerate sentencepiece sacrebleu peft bitsandbytes -q
!pip uninstall -y torchao

In [ ]:
import pandas as pd
import torch
import evaluate

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    DataCollatorForLanguageModeling,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainingArguments,
    Trainer
)

from peft import LoraConfig, get_peft_model

In [ ]:
data = pd.read_csv("/content/processed_data.csv")

qa_df = pd.DataFrame({
    "question": data["Question_for_QA"].astype(str),
    "answer": data["Answer_for_QA"].astype(str)
})

train_df, temp_df = train_test_split(qa_df,test_size=0.2,random_state=42,shuffle=True)
val_df, test_df = train_test_split(temp_df,test_size=0.5,random_state=42,shuffle=True)

train_data = Dataset.from_pandas(train_df.reset_index(drop=True))
val_data = Dataset.from_pandas(val_df.reset_index(drop=True))
test_data = Dataset.from_pandas(test_df.reset_index(drop=True))

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

Train: 3856
Validation: 482
Test: 483


In [ ]:
model_results = []

In [ ]:
t5_model_name = "google/mt5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name)

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
def preprocess_t5(examples):
    inputs = ["أجب عن السؤال التالي: " + q for q in examples["question"]]
    targets = [a for a in examples["answer"]]

    model_inputs = t5_tokenizer(inputs,max_length=128,truncation=True)
    labels = t5_tokenizer(text_target=targets,max_length=128,truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
t5_train = train_data.map(preprocess_t5, batched=True, remove_columns=train_data.column_names)
t5_val = val_data.map(preprocess_t5, batched=True, remove_columns=val_data.column_names)
t5_test = test_data.map(preprocess_t5, batched=True, remove_columns=test_data.column_names)

Map:   0%|          | 0/3856 [00:00<?, ? examples/s]

Map:   0%|          | 0/482 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

In [ ]:
t5_collator = DataCollatorForSeq2Seq(
    tokenizer=t5_tokenizer,
    model=t5_model
)

t5_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_arabic_qa",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    fp16=False
)

t5_trainer = Seq2SeqTrainer(
    model=t5_model,
    args=t5_args,
    train_dataset=t5_train,
    eval_dataset=t5_val,
    data_collator=t5_collator
)

In [ ]:
t5_train_output = t5_trainer.train()
t5_eval_output = t5_trainer.evaluate()

Epoch,Training Loss,Validation Loss
1,4.513838,3.104627
2,3.831678,2.880760
3,3.789126,2.826154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
3.789126,2.826154,3


In [ ]:
def generate_t5_answer(question):
    input_text = "أجب عن السؤال التالي: " + question

    inputs = t5_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(t5_model.device)

    outputs = t5_model.generate(
        **inputs,
        max_new_tokens=80,
        num_beams=5
    )

    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
for i in range(5):
    q = test_df.iloc[i]["question"]
    true_answer = test_df.iloc[i]["answer"]
    pred_answer = generate_t5_answer(q)

    print("="*100)
    print("Question:", q)
    print("True Answer:", true_answer)
    print("Predicted Answer:", pred_answer)

Question: ‏اعط مثالين عن الادوية القاتلة للخلايا او المثبطة للمناعة.
True Answer: الادوية القاتلة للخلايا مثل دوا السيكلوفوسفاميد، والادوية المثبطة للمناعة مثل دوا السيكلوسبورين تستخدم لعلاج انواع مختلفة من السرطان والامراض المناعية.
Predicted Answer: استخدم الادوية القاتلة للخلايا او المثبطة للمناعة.
Question: ما هي العوامل التي توثر علي التعليم العالي في الدول النامية؟
True Answer: العوامل التي توثر علي التعليم العالي في الدول النامية تشمل التمويل، جودة المناهج، وتوافر الكوادر التعليمية.
Predicted Answer: التكنولوجيا التي توثر علي التعليم العالي في الدول النامية هي العوامل التي توثر علي التعليم العالي في الدول النامية التي توثر علي التعليم العالي في الدول النامية.
Question: كيف يمكن تحسين جودة الهوا داخل المباني؟
True Answer: يمكن تحسين جودة الهوا داخل المباني باستخدام فلاتر هوا جيدة، التهوية الجيدة، وتقليل استخدام المواد الكيميايية الضارة.
Predicted Answer: امكن تحسين جودة الهوا داخل المباني، وتحسين جودة الهوا داخل المباني، وتحسين جودة الهوا داخل المباني، وتحسين جودة الهوا داخل المب

In [ ]:
bleu = evaluate.load("sacrebleu")

predictions = []
references = []

for example in test_data.select(range(min(100, len(test_data)))):
    prediction = generate_t5_answer(example["question"])
    predictions.append(prediction)
    references.append([example["answer"]])

t5_bleu = bleu.compute(predictions=predictions, references=references)["score"]

model_results.append({
    "Model": "mT5",
    "Train Loss": t5_train_output.training_loss,
    "Validation Loss": t5_eval_output["eval_loss"],
    "BLEU": t5_bleu
})

# Qwen

In [ ]:
qwen_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)

qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
def preprocess_qwen(examples):
    texts = []

    for q, a in zip(examples["question"], examples["answer"]):
        text = (
            "أجب عن السؤال التالي:\n"
            f"السؤال: {q}\n"
            f"الإجابة: {a}"
        )
        texts.append(text)

    inputs = qwen_tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

In [ ]:
qwen_train = train_data.map(preprocess_qwen, batched=True, remove_columns=train_data.column_names)
qwen_val = val_data.map(preprocess_qwen, batched=True, remove_columns=val_data.column_names)
qwen_test = test_data.map(preprocess_qwen, batched=True, remove_columns=test_data.column_names)

Map:   0%|          | 0/3856 [00:00<?, ? examples/s]

Map:   0%|          | 0/482 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

In [ ]:
lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

qwen_model = get_peft_model(qwen_model, lora_config)
qwen_model.print_trainable_parameters()

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


In [ ]:
qwen_collator = DataCollatorForLanguageModeling(
    tokenizer=qwen_tokenizer,
    mlm=False
)

qwen_args = TrainingArguments(
    output_dir="./qwen_arabic_qa",
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none"
)

qwen_trainer = Trainer(
    model=qwen_model,
    args=qwen_args,
    train_dataset=qwen_train,
    eval_dataset=qwen_val,
    data_collator=qwen_collator
)

In [ ]:
qwen_train_output = qwen_trainer.train()
qwen_eval_output = qwen_trainer.evaluate()

Epoch,Training Loss,Validation Loss
1,1.270238,1.260252
2,1.214282,1.220508
3,1.150668,1.209383


Training Loss,Validation Loss,Epoch
1.150668,1.209383,3


In [ ]:
def generate_qwen_answer(question):
    prompt = (
        "أجب عن السؤال التالي بنفس معنى الإجابة المتوقعة:\n"
        f"السؤال: {question}\n"
        "الإجابة:"
    )

    inputs = qwen_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(qwen_model.device)

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=40,
        num_beams=4,
        do_sample=False,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        early_stopping=True,
        pad_token_id=qwen_tokenizer.eos_token_id
    )

    generated_text = qwen_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text.split("الإجابة:")[-1].strip()

In [ ]:
for i in range(5):

    q = test_df.iloc[i]["question"]
    true_answer = test_df.iloc[i]["answer"]
    pred_answer = generate_qwen_answer(q)

    print("="*100)
    print("Question:", q)
    print("True Answer:", true_answer)
    print("Qwen Predicted Answer:", pred_answer)

Question: ‏اعط مثالين عن الادوية القاتلة للخلايا او المثبطة للمناعة.
True Answer: الادوية القاتلة للخلايا مثل دوا السيكلوفوسفاميد، والادوية المثبطة للمناعة مثل دوا السيكلوسبورين تستخدم لعلاج انواع مختلفة من السرطان والامراض المناعية.
Qwen Predicted Answer: المثال الأول: فيتامين C. المثال الثاني: مضادات الاكسدة الطبيعية.
Question: ما هي العوامل التي توثر علي التعليم العالي في الدول النامية؟
True Answer: العوامل التي توثر علي التعليم العالي في الدول النامية تشمل التمويل، جودة المناهج، وتوافر الكوادر التعليمية.
Qwen Predicted Answer: الامكانيات المادية، التحديات الاجتماعية، والسياسات التعليمية.
Question: كيف يمكن تحسين جودة الهوا داخل المباني؟
True Answer: يمكن تحسين جودة الهوا داخل المباني باستخدام فلاتر هوا جيدة، التهوية الجيدة، وتقليل استخدام المواد الكيميايية الضارة.
Qwen Predicted Answer: يمكن تحقيق ذلك من خلال استخدام مواد البناء ذات الجودة العالية، وتطبيق التقنيات الحديثة في التصميم والبناء، وتعزيز التدريب على إدارة
Question: اقترح نشاطا للاصدقا.
True Answer: يمكن تنظيم رحلات خارجي

In [ ]:
predictions = []
references = []

for example in test_data.select(range(min(100, len(test_data)))):
    prediction = generate_qwen_answer(example["question"])
    predictions.append(prediction)
    references.append([example["answer"]])

qwen_bleu = bleu.compute(predictions=predictions, references=references)["score"]

model_results.append({
    "Model": "Qwen",
    "Train Loss": qwen_train_output.training_loss,
    "Validation Loss": qwen_eval_output["eval_loss"],
    "BLEU": qwen_bleu
})

# GPT

In [ ]:
gpt_model_name = "aubmindlab/aragpt2-base"

gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_model_name)
gpt_model = AutoModelForCausalLM.from_pretrained(gpt_model_name)

gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def preprocess_gpt(examples):
    texts = []

    for q, a in zip(examples["question"], examples["answer"]):
        text = (
            "أجب عن السؤال التالي:\n"
            f"السؤال: {q}\n"
            f"الإجابة: {a}"
        )
        texts.append(text)

    inputs = gpt_tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

In [ ]:
gpt_train = train_data.map(preprocess_gpt, batched=True, remove_columns=train_data.column_names)
gpt_val = val_data.map(preprocess_gpt, batched=True, remove_columns=val_data.column_names)
gpt_test = test_data.map(preprocess_gpt, batched=True, remove_columns=test_data.column_names)

Map:   0%|          | 0/3856 [00:00<?, ? examples/s]

Map:   0%|          | 0/482 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

In [ ]:
gpt_collator = DataCollatorForLanguageModeling(
    tokenizer=gpt_tokenizer,
    mlm=False
)

gpt_args = TrainingArguments(
    output_dir="./aragpt2_arabic_qa",
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    fp16=False
)

gpt_trainer = Trainer(
    model=gpt_model,
    args=gpt_args,
    train_dataset=gpt_train,
    eval_dataset=gpt_val,
    data_collator=gpt_collator
)

In [ ]:
gpt_train_output = gpt_trainer.train()
gpt_eval_output = gpt_trainer.evaluate()

Epoch,Training Loss,Validation Loss
1,2.127187,1.749701
2,1.646125,1.639088
3,1.352966,1.619123


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
1.352966,1.619123,3


In [ ]:
def generate_gpt_answer(question):
    prompt = (
        "أجب عن السؤال التالي بنفس معنى الإجابة المتوقعة:\n"
        f"السؤال: {question}\n"
        "الإجابة:"
    )

    inputs = gpt_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(gpt_model.device)

    outputs = gpt_model.generate(
        **inputs,
        max_new_tokens=40,
        num_beams=4,
        do_sample=False,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        early_stopping=True,
        pad_token_id=gpt_tokenizer.eos_token_id
    )

    generated_text = gpt_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text.split("الإجابة:")[-1].strip()

In [ ]:
for i in range(5):

    q = test_df.iloc[i]["question"]
    true_answer = test_df.iloc[i]["answer"]
    pred_answer = generate_gpt_answer(q)

    print("="*100)
    print("Question:", q)
    print("True Answer:", true_answer)
    print("GPT Predicted Answer:", pred_answer)

Question: ‏اعط مثالين عن الادوية القاتلة للخلايا او المثبطة للمناعة.
True Answer: الادوية القاتلة للخلايا مثل دوا السيكلوفوسفاميد، والادوية المثبطة للمناعة مثل دوا السيكلوسبورين تستخدم لعلاج انواع مختلفة من السرطان والامراض المناعية.
GPT Predicted Answer: المثالين هما مضادات الاكسدة الطبيعية مثل فيتامين C ومضادات الاكسدة مثل فيتامين E ومضادات المناعة مثل فيتامين A ومضادات الربو. بالاضافة الي فيتامين E الذي يساعد في الوقاية من الامراض المزمنة مثل الربو
Question: ما هي العوامل التي توثر علي التعليم العالي في الدول النامية؟
True Answer: العوامل التي توثر علي التعليم العالي في الدول النامية تشمل التمويل، جودة المناهج، وتوافر الكوادر التعليمية.
GPT Predicted Answer: العوامل تشمل الفقر، البطالة، والسياسات الحكومية. تشمل السياسات التعليمية، التعليم العالي، والتعليم العالي. سياسات تعليمية محفزة، ودعم البحث العلمي. بالاضافة الي السياسات الحكومية الداعمة للتعليم العالي
Question: كيف يمكن تحسين جودة الهوا داخل المباني؟
True Answer: يمكن تحسين جودة الهوا داخل المباني باستخدام فلاتر هوا جيدة، التهو

In [ ]:
predictions = []
references = []

for example in test_data.select(range(min(100, len(test_data)))):
    prediction = generate_gpt_answer(example["question"])
    predictions.append(prediction)
    references.append([example["answer"]])

gpt_bleu = bleu.compute(predictions=predictions, references=references)["score"]

model_results.append({
    "Model": "AraGPT2",
    "Train Loss": gpt_train_output.training_loss,
    "Validation Loss": gpt_eval_output["eval_loss"],
    "BLEU": gpt_bleu
})

In [ ]:
results_df = pd.DataFrame(model_results)
results_df

,Model,Train Loss,Validation Loss,BLEU
0,mT5,5.160455,2.826154,13.539193
1,Qwen,1.253190,1.209383,2.729197
2,AraGPT2,1.890731,1.619123,4.740046


In [ ]:
best_model = results_df.sort_values(by="BLEU", ascending=False).iloc[0]

print("Best Model:", best_model["Model"])
print("Best BLEU Score:", best_model["BLEU"])

Best Model: mT5
Best BLEU Score: 13.539192634482383


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
t5_model.save_pretrained("/content/drive/MyDrive/best_qa_model")
t5_tokenizer.save_pretrained("/content/drive/MyDrive/best_qa_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/best_qa_model/tokenizer_config.json',
 '/content/drive/MyDrive/best_qa_model/tokenizer.json')

Three transformer based models were fine-tuned for Arabic Question Answering: mT5, Qwen and AraGPT2.
All models were trained using the same train/validation/test split, maximum sequence length of 128, learning rate of 5e-5 and 3 epochs.
The models were evaluated using BLEU score.
mT5 achieved the highest BLEU score and was selected as the final model for deployment

# Seq to Seq

In [ ]:
import numpy as np
import pandas as pd


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, GRU, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

In [ ]:
MAX_LEN = 64
DEC_LEN = 63
EMBEDDING_DIM = 300

data = pd.read_csv("/content/processed_data.csv")

questions = data["Question_for_QA"].astype(str).tolist()
answers = data["Answer_for_QA"].astype(str).tolist()

target_texts = ["starttoken " + ans + " endtoken" for ans in answers]

In [ ]:
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(questions)

input_sequences = input_tokenizer.texts_to_sequences(questions)
input_padded = pad_sequences(input_sequences, maxlen=MAX_LEN, padding="post", truncating="post")

input_vocab_size = len(input_tokenizer.word_index) + 1

In [ ]:
target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)

target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_padded = pad_sequences(target_sequences, maxlen=MAX_LEN, padding="post", truncating="post")

decoder_input_tokens = target_padded[:, :-1]
decoder_target_tokens = target_padded[:, 1:]
decoder_target_tokens = decoder_target_tokens.reshape(decoder_target_tokens.shape[0], decoder_target_tokens.shape[1], 1)

decoder_vocab_size = len(target_tokenizer.word_index) + 1

In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653909 sha256=eda979877737daff1e03b1a5342d3cc4ca3e4985ad06c912b785742bc6e6154b
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
import fasttext

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz
!gunzip cc.ar.300.bin.gz

--2026-06-09 01:39:16--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.239.50.18, 18.239.50.120, 18.239.50.9, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.239.50.18|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4500982519 (4.2G) [application/octet-stream]
Saving to: ‘cc.ar.300.bin.gz’

cc.ar.300.bin.gz    100%[===================>]   4.19G   152MB/s    in 32s     

2026-06-09 01:39:48 (135 MB/s) - ‘cc.ar.300.bin.gz’ saved [4500982519/4500982519]



In [ ]:
fasttext_model = fasttext.load_model("cc.ar.300.bin")

In [ ]:
encoder_embedding_matrix = np.zeros((input_vocab_size, EMBEDDING_DIM))

for word, idx in input_tokenizer.word_index.items():
    encoder_embedding_matrix[idx] = fasttext_model.get_word_vector(word)

In [ ]:
decoder_embedding_matrix = np.zeros((decoder_vocab_size, EMBEDDING_DIM))

for word, idx in target_tokenizer.word_index.items():
    decoder_embedding_matrix[idx] = fasttext_model.get_word_vector(word)

In [ ]:
X_train, X_temp, dec_in_train, dec_in_temp, dec_tar_train, dec_tar_temp = train_test_split(
    input_padded,
    decoder_input_tokens,
    decoder_target_tokens,
    test_size=0.2,
    random_state=42
)

X_val, X_test, dec_in_val, dec_in_test, dec_tar_val, dec_tar_test = train_test_split(
    X_temp,
    dec_in_temp,
    dec_tar_temp,
    test_size=0.5,
    random_state=42
)

In [ ]:
HIDDEN_UNITS = 256

encoder_inputs = Input(shape=(MAX_LEN,))

encoder_embedding = Embedding(
    input_dim=input_vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[encoder_embedding_matrix],
    trainable=False,
    mask_zero=True
)(encoder_inputs)

encoder_gru = GRU(HIDDEN_UNITS,return_state=True,name="encoder_gru")
_, encoder_state = encoder_gru(encoder_embedding)

decoder_inputs = Input(shape=(None,))

decoder_embedding_layer = Embedding(
    input_dim=decoder_vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[decoder_embedding_matrix],
    trainable=False,
    mask_zero=True
)

decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_gru = GRU(HIDDEN_UNITS,return_sequences=True,return_state=True,name="decoder_gru")
decoder_outputs, _ = decoder_gru(decoder_embedding,initial_state=encoder_state)
decoder_outputs = Dense(
    decoder_vocab_size,
    activation="softmax"
)(decoder_outputs)

fasttext_gru_model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

fasttext_gru_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fasttext_gru_model.summary()

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12      │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_13      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_10        │ (None, 64, 300)   │  2,092,500 │ input_layer_12[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_12        │ (None, 64)        │          0 │ input_layer_12[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_11        │ (None, None, 300) │  4,792,200 │ input_layer_13[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_gru (GRU)   │ [(None, 256),     │    428,544 │ embedding_10[0][… │
│                     │ (None, 256)]      │            │ not_equal_12[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_gru (GRU)   │ [(None, None,     │    428,544 │ embedding_11[0][… │
│                     │ 256), (None,      │            │ encoder_gru[0][1] │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, None,      │  4,105,318 │ decoder_gru[0][0] │
│                     │ 15974)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 11,847,106 (45.19 MB)

 Trainable params: 4,962,406 (18.93 MB)

 Non-trainable params: 6,884,700 (26.26 MB)

In [ ]:
history = fasttext_gru_model.fit(
    [X_train, dec_in_train],
    dec_tar_train,
    batch_size=32,
    epochs=10,
    validation_data=([X_val, dec_in_val], dec_tar_val)
)

Epoch 1/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 12s 85ms/step - accuracy: 0.0682 - loss: 8.3458 - val_accuracy: 0.1001 - val_loss: 7.9099
Epoch 2/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 10s 81ms/step - accuracy: 0.1113 - loss: 7.5279 - val_accuracy: 0.1281 - val_loss: 7.6170
Epoch 3/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 10s 81ms/step - accuracy: 0.1305 - loss: 7.0855 - val_accuracy: 0.1394 - val_loss: 7.3268
Epoch 4/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 10s 80ms/step - accuracy: 0.1492 - loss: 6.5548 - val_accuracy: 0.1664 - val_loss: 6.9158
Epoch 5/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 10s 82ms/step - accuracy: 0.1777 - loss: 5.9880 - val_accuracy: 0.1954 - val_loss: 6.6457
Epoch 6/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 10s 82ms/step - accuracy: 0.2054 - loss: 5.4712 - val_accuracy: 0.2151 - val_loss: 6.4264
Epoch 7/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 10s 82ms/step - accuracy: 0.2319 - loss: 5.0129 - val_accuracy: 0.2365 - val_loss: 6.2769
Epoch 8/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 10s 81ms/step - accuracy: 0.2580 - loss: 4.5935 - 

In [ ]:
gru_eval = fasttext_gru_model.evaluate(
    [X_val, dec_in_val],
    dec_tar_val,
    verbose=0
)

print("Validation Loss:", gru_eval[0])
print("Validation Accuracy:", gru_eval[1])

Validation Loss: 5.982315540313721
Validation Accuracy: 0.275770366191864


In [ ]:
gru_val_loss = gru_eval[0]
gru_val_acc = gru_eval[1]

# LSTM

In [ ]:
from tensorflow.keras.layers import LSTM

HIDDEN_UNITS = 256

# Encoder
encoder_inputs = Input(shape=(MAX_LEN,))

encoder_embedding = Embedding(
    input_dim=input_vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[encoder_embedding_matrix],
    trainable=False,
    mask_zero=True
)(encoder_inputs)

encoder_lstm = LSTM(
    HIDDEN_UNITS,
    return_state=True,
    name="encoder_lstm"
)

_, state_h, state_c = encoder_lstm(encoder_embedding)

# Decoder
decoder_inputs = Input(shape=(None,))

decoder_embedding_layer = Embedding(
    input_dim=decoder_vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[decoder_embedding_matrix],
    trainable=False,
    mask_zero=True
)

decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    HIDDEN_UNITS,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=[state_h, state_c]
)

decoder_outputs = Dense(
    decoder_vocab_size,
    activation="softmax"
)(decoder_outputs)

fasttext_lstm_model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

fasttext_lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fasttext_lstm_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_5       │ (None, 63)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 64, 300)   │  2,092,500 │ input_layer_4[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 64)        │          0 │ input_layer_4[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 63, 300)   │  4,792,200 │ input_layer_5[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │    219,648 │ embedding_4[0][0… │
│                     │ (None, 128),      │            │ not_equal_4[0][0] │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 63, 128), │    219,648 │ embedding_5[0][0… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 63, 15974) │  2,060,646 │ decoder_lstm[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,384,642 (35.80 MB)

 Trainable params: 2,499,942 (9.54 MB)

 Non-trainable params: 6,884,700 (26.26 MB)

In [ ]:
history_lstm = fasttext_lstm_model.fit(
    [X_train, dec_in_train],
    dec_tar_train,
    batch_size=32,
    epochs=10,
    validation_data=([X_val, dec_in_val], dec_tar_val)
)

Epoch 1/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 12s 76ms/step - accuracy: 0.0564 - loss: 8.3185 - val_accuracy: 0.0618 - val_loss: 7.8685
Epoch 2/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 9s 72ms/step - accuracy: 0.0679 - loss: 7.6248 - val_accuracy: 0.0725 - val_loss: 7.7752
Epoch 3/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 9s 72ms/step - accuracy: 0.0959 - loss: 7.4617 - val_accuracy: 0.1233 - val_loss: 7.7043
Epoch 4/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.1242 - loss: 7.3201 - val_accuracy: 0.1288 - val_loss: 7.6226
Epoch 5/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 9s 71ms/step - accuracy: 0.1263 - loss: 7.1841 - val_accuracy: 0.1306 - val_loss: 7.5392
Epoch 6/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 9s 74ms/step - accuracy: 0.1287 - loss: 7.0506 - val_accuracy: 0.1337 - val_loss: 7.4571
Epoch 7/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.1354 - loss: 6.8935 - val_accuracy: 0.1436 - val_loss: 7.3346
Epoch 8/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 8s 70ms/step - accuracy: 0.1444 - loss: 6.7129 - val_acc

In [ ]:
lstm_eval = fasttext_lstm_model.evaluate(
    [X_val, dec_in_val],
    dec_tar_val,
    verbose=0
)

print("Validation Loss:", lstm_eval[0])
print("Validation Accuracy:", lstm_eval[1])

Validation Loss: 7.032784461975098
Validation Accuracy: 0.16736866533756256


In [ ]:
lstm_val_loss = lstm_eval[0]
lstm_val_acc = lstm_eval[1]

# Simple RNN

In [ ]:
from tensorflow.keras.layers import SimpleRNN

HIDDEN_UNITS = 256

# Encoder
encoder_inputs = Input(shape=(MAX_LEN,))

encoder_embedding = Embedding(
    input_dim=input_vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[encoder_embedding_matrix],
    trainable=False,
    mask_zero=True
)(encoder_inputs)

encoder_rnn = SimpleRNN(
    HIDDEN_UNITS,
    return_state=True,
    name="encoder_rnn"
)

_, encoder_state = encoder_rnn(encoder_embedding)

# Decoder
decoder_inputs = Input(shape=(None,))

decoder_embedding_layer = Embedding(
    input_dim=decoder_vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[decoder_embedding_matrix],
    trainable=False,
    mask_zero=True
)

decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_rnn = SimpleRNN(
    HIDDEN_UNITS,
    return_sequences=True,
    return_state=True,
    name="decoder_rnn"
)

decoder_outputs, _ = decoder_rnn(
    decoder_embedding,
    initial_state=encoder_state
)

decoder_outputs = Dense(
    decoder_vocab_size,
    activation="softmax"
)(decoder_outputs)

fasttext_rnn_model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

fasttext_rnn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fasttext_rnn_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, 63)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 64, 300)   │  2,092,500 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_6         │ (None, 64)        │          0 │ input_layer_6[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, 63, 300)   │  4,792,200 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_rnn         │ [(None, 128),     │     54,912 │ embedding_6[0][0… │
│ (SimpleRNN)         │ (None, 128)]      │            │ not_equal_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_rnn         │ [(None, 63, 128), │     54,912 │ embedding_7[0][0… │
│ (SimpleRNN)         │ (None, 128)]      │            │ encoder_rnn[0][1] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 63, 15974) │  2,060,646 │ decoder_rnn[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,055,170 (34.54 MB)

 Trainable params: 2,170,470 (8.28 MB)

 Non-trainable params: 6,884,700 (26.26 MB)

In [ ]:
history_rnn = fasttext_rnn_model.fit(
    [X_train, dec_in_train],
    dec_tar_train,
    batch_size=32,
    epochs=10,
    validation_data=([X_val, dec_in_val], dec_tar_val)
)

Epoch 1/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 15s 65ms/step - accuracy: 0.0593 - loss: 8.2691 - val_accuracy: 0.1066 - val_loss: 7.8975
Epoch 2/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.0919 - loss: 7.6015 - val_accuracy: 0.1247 - val_loss: 7.6591
Epoch 3/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.1253 - loss: 7.2977 - val_accuracy: 0.1295 - val_loss: 7.5000
Epoch 4/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.1296 - loss: 7.0345 - val_accuracy: 0.1356 - val_loss: 7.3492
Epoch 5/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.1359 - loss: 6.7454 - val_accuracy: 0.1427 - val_loss: 7.1639
Epoch 6/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.1427 - loss: 6.4617 - val_accuracy: 0.1524 - val_loss: 7.0136
Epoch 7/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.1512 - loss: 6.2081 - val_accuracy: 0.1578 - val_loss: 6.9186
Epoch 8/10
121/121 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.1608 - loss: 5.9606 - val_acc

In [ ]:
rnn_eval = fasttext_rnn_model.evaluate(
    [X_val, dec_in_val],
    dec_tar_val,
    verbose=0
)

print("Validation Loss:", rnn_eval[0])
print("Validation Accuracy:", rnn_eval[1])

Validation Loss: 6.637207508087158
Validation Accuracy: 0.19142098724842072


In [ ]:
rnn_val_loss = rnn_eval[0]
rnn_val_acc = rnn_eval[1]

In [ ]:
results = pd.DataFrame({
    "Model": [
        "FastText + SimpleRNN",
        "FastText + LSTM",
        "FastText + GRU"
    ],
    "Validation Loss": [
        rnn_val_loss,
        lstm_val_loss,
        gru_val_loss
    ],
    "Validation Accuracy": [
        rnn_val_acc,
        lstm_val_acc,
        gru_val_acc
    ]
})

results

,Model,Validation Loss,Validation Accuracy
0,FastText + SimpleRNN,6.637208,0.191421
1,FastText + LSTM,7.032784,0.167369
2,FastText + GRU,5.982316,0.275770


In [ ]:
# Encoder inference model
encoder_model = Model(
    encoder_inputs,
    encoder_state
)

# Decoder inference model
decoder_state_input = Input(shape=(HIDDEN_UNITS,))

decoder_inf_embedding = decoder_embedding_layer(decoder_inputs)

decoder_inf_outputs, decoder_inf_state = decoder_gru(
    decoder_inf_embedding,
    initial_state=decoder_state_input
)

decoder_inf_outputs = fasttext_gru_model.layers[-1](decoder_inf_outputs)

decoder_model = Model(
    [decoder_inputs, decoder_state_input],
    [decoder_inf_outputs, decoder_inf_state]
)

In [ ]:
reverse_target_word_index = {
    idx: word for word, idx in target_tokenizer.word_index.items()
}

start_token_id = target_tokenizer.word_index["starttoken"]
end_token_id = target_tokenizer.word_index["endtoken"]

def decode_sequence(input_seq):
    state = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = start_token_id

    decoded_words = []

    for _ in range(DEC_LEN):
        output_tokens, state = decoder_model.predict(
            [target_seq, state],
            verbose=0
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index, "")

        if sampled_word == "endtoken":
            break

        if sampled_word != "":
            decoded_words.append(sampled_word)

        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

    return " ".join(decoded_words)

In [ ]:
def answer_question(question):
    seq = input_tokenizer.texts_to_sequences([question])
    padded = pad_sequences(
        seq,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

    answer = decode_sequence(padded)
    return answer

In [ ]:
question = "ما هي أعراض مرض السكري؟"
print(answer_question(question))

اسباب تشقق الدم تشمل عوامل مثل التغيرات مثل جبنة الشيفر او عوامل في تحسين الطاقة، وتحسين صحة القلب، وتعزيز الصحة العقلية،


In [ ]:
!pip install nltk

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

predictions = []
references = []

for i in range(len(X_test)):
    input_seq = X_test[i].reshape(1, MAX_LEN)

    predicted_answer = decode_sequence(input_seq)

    true_ids = dec_tar_test[i].reshape(-1)

    true_words = []
    for token_id in true_ids:
        if token_id == 0:
            continue

        word = reverse_target_word_index.get(int(token_id), "")

        if word == "endtoken":
            break

        if word != "starttoken" and word != "":
            true_words.append(word)

    true_answer = " ".join(true_words)

    predictions.append(predicted_answer.split())
    references.append([true_answer.split()])

In [ ]:
smooth = SmoothingFunction().method1

bleu_score = corpus_bleu(
    references,
    predictions,
    smoothing_function=smooth
)

print("BLEU Score:", bleu_score)

BLEU Score: 0.0247096246535976
